In [1]:
# -*- coding: utf-8 -*-
"""
第一部分：自适应 EMD 二分类数据预处理（min_corr=0.80，无标签泄露版）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(2024)
np.random.seed(2024)

import pandas as pd
import matplotlib.pyplot as plt
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


class AdaptiveEMDProcessor:
    def __init__(self, n_imfs=8, energy_threshold=0.02, corr_threshold=0.12, min_corr=0.80):
        self.n_imfs = n_imfs
        self.emd = EMD()
        self.energy_threshold = energy_threshold
        self.corr_threshold = corr_threshold
        self.min_corr = min_corr

    def _calculate_imf_metrics(self, imfs, orig_signal):
        total_energy = np.sum(orig_signal ** 2)
        energy_ratios = []
        correlations = []

        for imf in imfs:
            imf_energy = np.sum(imf ** 2)
            energy_ratios.append(imf_energy / total_energy)

            if np.std(imf) < 1e-6 or np.std(orig_signal) < 1e-6:
                correlations.append(0)
            else:
                corr = np.corrcoef(orig_signal, imf)[0, 1]
                correlations.append(corr)
        return np.array(energy_ratios), np.array(correlations)

    def _calc_correlation(self, orig, recon):
        min_len = min(len(orig), len(recon))
        if min_len < 2:
            return 0.0
        orig, recon = orig[:min_len], recon[:min_len]
        if np.std(orig) < 1e-6 or np.std(recon) < 1e-6:
            return 0.0
        return np.corrcoef(orig, recon)[0, 1]

    def _adaptive_denoise(self, sample, is_theft_sample=False):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2:
                return orig_signal

            energy_ratios, correlations = self._calculate_imf_metrics(imfs, orig_signal)

            # 第一步：原始逻辑筛选
            if is_theft_sample:
                useful_mask = (energy_ratios > self.energy_threshold * 0.8) | \
                              (np.abs(correlations) > self.corr_threshold * 0.8)
            else:
                useful_mask = (energy_ratios > self.energy_threshold) | \
                              (np.abs(correlations) > self.corr_threshold)
            useful_imfs = imfs[useful_mask]

            # 过滤逻辑
            if not is_theft_sample and len(useful_imfs) > 1:
                useful_imfs = useful_imfs[1:]
            elif is_theft_sample and len(useful_imfs) > 1:
                first_imf_energy = np.sum(useful_imfs[0] ** 2) / np.sum(orig_signal ** 2)
                if first_imf_energy < 0.015:
                    useful_imfs = useful_imfs[1:]

            reconstructed = np.sum(useful_imfs, axis=0) + residue
            recon_corr = self._calc_correlation(orig_signal, reconstructed)

            # 第二步：不达标则放宽筛选（降至50%）
            if recon_corr < self.min_corr:
                useful_mask = (energy_ratios > self.energy_threshold * 0.5) | \
                              (np.abs(correlations) > self.corr_threshold * 0.5)
                useful_imfs = imfs[useful_mask]
                reconstructed = np.sum(useful_imfs, axis=0) + residue
                recon_corr = self._calc_correlation(orig_signal, reconstructed)

                # 第三步：仍不达标则强制保留所有IMF
                if recon_corr < self.min_corr:
                    useful_imfs = imfs
                    reconstructed = np.sum(useful_imfs, axis=0) + residue

            # 幅值校准
            if np.max(reconstructed) != 0 and np.max(orig_signal) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            if len(reconstructed) > len(orig_signal):
                reconstructed = reconstructed[:len(orig_signal)]

            return reconstructed
        except Exception:
            return orig_signal

    def process_data(self, X_data, y_data=None):
        print(f"=== 自适应 EMD 处理 {len(X_data)} 个样本 ===")
        processed_signals = []
        for sample_idx, sample in enumerate(X_data):
            if sample_idx % 100 == 0:
                print(f"处理进度: {sample_idx}/{len(X_data)}")

            is_theft = False
            if y_data is not None and y_data[sample_idx] == 1:
                is_theft = True

            processed = self._adaptive_denoise(sample, is_theft_sample=is_theft)
            processed_signals.append(processed)
        return np.array(processed_signals) if y_data is None else (np.array(processed_signals), y_data)


def main():
    print("=== 第一阶段：二分类 EMD 自适应去噪分解 (min_corr=0.80) ===")

    # 请改成你真实的数据集路径
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'

    try:
        df = pd.read_excel(file_path)
    except Exception:
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except Exception:
            df = pd.read_csv(file_path, encoding='ANSI')

    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)

    Y = df['output'].values
    X = df.drop(['output'], axis=1).values

    print(f"原始标签分布，正常样本(0): {np.sum(Y == 0)}, 窃电样本(1): {np.sum(Y == 1)}")

    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )

    processor = AdaptiveEMDProcessor(min_corr=0.80)

    # 训练集：传入标签（训练阶段允许自适应）
    X_train, y_train = processor.process_data(X_train_raw, y_train_raw)

    # 测试集：不传标签，避免标签泄露（测试阶段固定策略）
    print("\n=== 测试集预处理（不传入标签）===")
    X_test = processor.process_data(X_test_raw)
    y_test = y_test_raw

    # 保存结果
    output_dir = "adaptive_emd_binary_0.8011"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test)

    print(f"\n✅ EMD 分解预处理完成！纯净数据已保存至: {output_dir}")
    print(f"   训练集: X={X_train.shape}, y={y_train.shape}")
    print(f"   测试集: X={X_test.shape}, y={y_test.shape}")


if __name__ == "__main__":
    main()

=== 第一阶段：二分类 EMD 自适应去噪分解 (min_corr=0.80) ===
原始标签分布，正常样本(0): 10846, 窃电样本(1): 3615
=== 自适应 EMD 处理 11568 个样本 ===
处理进度: 0/11568
处理进度: 100/11568
处理进度: 200/11568
处理进度: 300/11568
处理进度: 400/11568
处理进度: 500/11568
处理进度: 600/11568
处理进度: 700/11568
处理进度: 800/11568
处理进度: 900/11568
处理进度: 1000/11568
处理进度: 1100/11568
处理进度: 1200/11568
处理进度: 1300/11568
处理进度: 1400/11568
处理进度: 1500/11568
处理进度: 1600/11568
处理进度: 1700/11568
处理进度: 1800/11568
处理进度: 1900/11568
处理进度: 2000/11568
处理进度: 2100/11568
处理进度: 2200/11568
处理进度: 2300/11568
处理进度: 2400/11568
处理进度: 2500/11568
处理进度: 2600/11568
处理进度: 2700/11568
处理进度: 2800/11568
处理进度: 2900/11568
处理进度: 3000/11568
处理进度: 3100/11568
处理进度: 3200/11568
处理进度: 3300/11568
处理进度: 3400/11568
处理进度: 3500/11568
处理进度: 3600/11568
处理进度: 3700/11568
处理进度: 3800/11568
处理进度: 3900/11568
处理进度: 4000/11568
处理进度: 4100/11568
处理进度: 4200/11568
处理进度: 4300/11568
处理进度: 4400/11568
处理进度: 4500/11568
处理进度: 4600/11568
处理进度: 4700/11568
处理进度: 4800/11568
处理进度: 4900/11568
处理进度: 5000/11568
处理进度: 5100/11568
处理进度: 5200/11568


In [3]:
# -*- coding: utf-8 -*-
"""
第一部分：标准EMD二分类数据预处理（无任何阈值筛选，全保留）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(2024)
np.random.seed(2024)

import pandas as pd
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class StandardEMDProcessor:
    def __init__(self, n_imfs=8):
        self.n_imfs = n_imfs
        self.emd = EMD()

    def _standard_denoise(self, sample):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2: 
                return orig_signal
            
            # 标准EMD：不筛选，全部保留，直接相加
            reconstructed = np.sum(imfs, axis=0) + residue
            
            # 幅值与长度校准
            if np.max(reconstructed) != 0 and np.max(orig_signal) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            if len(reconstructed) > len(orig_signal):
                reconstructed = reconstructed[:len(orig_signal)]
                
            return reconstructed
        except Exception:
            return orig_signal

    def process_data(self, X_data):
        print(f"=== 标准 EMD 处理 {len(X_data)} 个样本 ===")
        processed_signals = []
        for sample_idx, sample in enumerate(X_data):
            if sample_idx % 100 == 0:
                print(f"处理进度: {sample_idx}/{len(X_data)}")
            processed_signals.append(self._standard_denoise(sample))
        return np.array(processed_signals)

def main():
    print("=== 第一阶段：标准 EMD 分解预处理 ===")
    
    # ⚠️ 请确保路径与你的主模型完全一致
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    
    try:
        df = pd.read_excel(file_path)
    except Exception:
        try:
            df = pd.read_csv(file_path, encoding='utf-8')
        except Exception:
            df = pd.read_csv(file_path, encoding='ANSI')

    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values
    
    print(f"原始标签分布，正常样本(0): {np.sum(Y==0)}, 窃电样本(1): {np.sum(Y==1)}")
    
    # ⚠️ 必须与主模型、固定阈值模型完全一致的切分比例 (test_size=0.2 还是 0.4，请统一)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )
    
    processor = StandardEMDProcessor(n_imfs=8)
    X_train = processor.process_data(X_train_raw)
    X_test = processor.process_data(X_test_raw)
    
    output_dir = "standard_emd_data"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train_raw)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test_raw)
    
    print(f"\n✅ 标准 EMD 分解预处理完成！数据已保存至: {output_dir}")

if __name__ == "__main__":
    main()

=== 第一阶段：标准 EMD 分解预处理 ===
原始标签分布，正常样本(0): 10846, 窃电样本(1): 3615
=== 标准 EMD 处理 11568 个样本 ===
处理进度: 0/11568
处理进度: 100/11568
处理进度: 200/11568
处理进度: 300/11568
处理进度: 400/11568
处理进度: 500/11568
处理进度: 600/11568
处理进度: 700/11568
处理进度: 800/11568
处理进度: 900/11568
处理进度: 1000/11568
处理进度: 1100/11568
处理进度: 1200/11568
处理进度: 1300/11568
处理进度: 1400/11568
处理进度: 1500/11568
处理进度: 1600/11568
处理进度: 1700/11568
处理进度: 1800/11568
处理进度: 1900/11568
处理进度: 2000/11568
处理进度: 2100/11568
处理进度: 2200/11568
处理进度: 2300/11568
处理进度: 2400/11568
处理进度: 2500/11568
处理进度: 2600/11568
处理进度: 2700/11568
处理进度: 2800/11568
处理进度: 2900/11568
处理进度: 3000/11568
处理进度: 3100/11568
处理进度: 3200/11568
处理进度: 3300/11568
处理进度: 3400/11568
处理进度: 3500/11568
处理进度: 3600/11568
处理进度: 3700/11568
处理进度: 3800/11568
处理进度: 3900/11568
处理进度: 4000/11568
处理进度: 4100/11568
处理进度: 4200/11568
处理进度: 4300/11568
处理进度: 4400/11568
处理进度: 4500/11568
处理进度: 4600/11568
处理进度: 4700/11568
处理进度: 4800/11568
处理进度: 4900/11568
处理进度: 5000/11568
处理进度: 5100/11568
处理进度: 5200/11568
处理进度: 5300/11568
处理进

In [7]:
# -*- coding: utf-8 -*-
"""
第二部分：标准EMD-SE-CNN 二分类网络训练与预测
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(2024)
np.random.seed(2024)
tf.random.set_seed(2024)

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=3, **kwargs):
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
        
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False, kernel_initializer='he_normal')
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')
        super(SEBlock, self).build(input_shape)
        
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])
    
    def get_config(self):
        config = super(SEBlock, self).get_config()
        config.update({'reduction': self.reduction})
        return config

def main():
    print("=== 第二阶段：标准 EMD-SE-CNN 二分类窃电检测 ===")
    
    data_dir = "standard_emd_data"
    try:
        X_train = np.load(os.path.join(data_dir, "X_train.npy"))
        y_train = np.load(os.path.join(data_dir, "y_train.npy"))
        X_test = np.load(os.path.join(data_dir, "X_test.npy"))
        y_test = np.load(os.path.join(data_dir, "y_test.npy"))
        
        X_train = X_train[..., np.newaxis] 
        X_test = X_test[..., np.newaxis]   
        print(f"✅ 数据加载成功！训练集: {X_train.shape}, 测试集: {X_test.shape}")
    except FileNotFoundError:
        print("❌ 找不到数据文件夹，请先运行第一部分预处理脚本！")
        return
    
    inputs = Input(shape=X_train.shape[1:]) 
    
    x = Conv1D(32, 5, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Conv1D(64, 3, strides=1, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(reduction=3)(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2, padding='same')(x)
    x = Dropout(0.3)(x)
    
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = Dropout(0.4)(x)
    
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        loss='binary_crossentropy',
        optimizer=Adam(learning_rate=0.0005),
        metrics=['accuracy']
    )
    
    early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)
    
    print("\n🚀 开始模型训练...")
    history = model.fit(
        X_train, y_train,
        epochs=150,
        batch_size=32,
        validation_data=(X_test, y_test), 
        callbacks=[early_stop, reduce_lr],
        shuffle=True,
        verbose=1
    )
    
    print("\n=== 模型硬核指标评估 ===")
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr_curve, tpr_curve)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr_val = fp / (fp + tn + 1e-10)
    fnr_val = fn / (fn + tp + 1e-10)
    g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
    
    print("\n" + "="*50)
    print("=== SCI 级二分类详细性能指标 (标准EMD) ===")
    print("="*50)
    print(f"准确率 (Accuracy)       : {accuracy:.4f}")
    print(f"精确率 (Precision)      : {precision:.4f}")
    print(f"召回率 (Recall)         : {recall:.4f}")
    print(f"F1分数 (F1-Score)       : {f1:.4f}")
    print(f"AUC面积 (ROC-AUC)       : {roc_auc:.4f}")
    print("-" * 50)
    print(f"误报率 (FPR)            : {fpr_val:.4f}")
    print(f"漏报率 (FNR)            : {fnr_val:.4f}")
    print(f"综合均值 (G-Mean)       : {g_mean:.4f}")
    print(f"马修斯系数 (MCC)        : {mcc:.4f}")
    print("="*50)
    
    print("\n分类报告:")
    print(classification_report(y_test, y_pred, target_names=['Normal (0)', 'Theft (1)'], digits=4))
    
    save_path = "Binary_Results_CSV_Standard_EMD2"
    os.makedirs(save_path, exist_ok=True)
    df_save = pd.DataFrame({'label': y_test, 'pred': y_pred_proba})
    csv_filename = os.path.join(save_path, "Binary_Standard_EMD_CNN_SE2.csv")
    df_save.to_csv(csv_filename, index=False)
    print(f"\n✅ 预测结果已成功导出至: {csv_filename}")

if __name__ == "__main__":
    main()

=== 第二阶段：标准 EMD-SE-CNN 二分类窃电检测 ===
✅ 数据加载成功！训练集: (11568, 1034, 1), 测试集: (2893, 1034, 1)

🚀 开始模型训练...
Epoch 1/150
362/362 [==============================] - 37s 96ms/step - loss: 0.5023 - accuracy: 0.8337 - val_loss: 0.6490 - val_accuracy: 0.7501 - lr: 5.0000e-04
Epoch 2/150
362/362 [==============================] - 36s 100ms/step - loss: 0.4260 - accuracy: 0.8678 - val_loss: 0.4470 - val_accuracy: 0.8807 - lr: 5.0000e-04
Epoch 3/150
362/362 [==============================] - 36s 100ms/step - loss: 0.4081 - accuracy: 0.8836 - val_loss: 0.3975 - val_accuracy: 0.8863 - lr: 5.0000e-04
Epoch 4/150
362/362 [==============================] - 37s 101ms/step - loss: 0.3886 - accuracy: 0.8896 - val_loss: 0.3888 - val_accuracy: 0.8956 - lr: 5.0000e-04
Epoch 5/150
362/362 [==============================] - 37s 102ms/step - loss: 0.3791 - accuracy: 0.8976 - val_loss: 0.3973 - val_accuracy: 0.9018 - lr: 5.0000e-04
Epoch 6/150
362/362 [==============================] - 36s 101ms/step - loss: 0.3665 

In [5]:
# -*- coding: utf-8 -*-
"""
【消融实验专用】轻量级 Transformer 模型（严格保持除网络结构外的所有参数一致）
目标：在相同的训练、数据切分、评估标准下，对比 CNN 与 Transformer 的性能
"""
import os
# 1. 严格锁死全局随机种子，确保与 CNN 在同一起跑线
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

import matplotlib
matplotlib.rcParams["font.family"] = ["Times New Roman", "SimHei", "Arial"]
matplotlib.rcParams["axes.unicode_minus"] = False
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout, BatchNormalization, Input
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, GlobalAveragePooling1D, Embedding, AveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print("=== 消融实验：轻量级 Transformer 模型 (CPU极速版) ===")

# 2. 数据加载（严格一致）
print("\n1. 加载数据...")
file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'

try:
    df = pd.read_excel(file_path)
except Exception:
    df = pd.read_csv(file_path, encoding='utf-8')

if 'Unnamed: 0' in df.columns:
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    
Y = df['output'].astype(int).values
X = df.drop(['output'], axis=1).values

# 保持一致的归一化逻辑
scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X).astype(np.float32)

# 3. 严格对齐数据集划分 (test_size=0.2, stratify=Y, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    X_scaled, Y, 
    test_size=0.2,     
    random_state=42,
    shuffle=True,      
    stratify=Y         
)

# 调整维度
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]
input_dim = x_train.shape[1]

# 4. 构建轻量级 Transformer 模型
inputs = Input(shape=(input_dim, 1))

# 【核心加速】先做一维平均池化，把 1034 个时间步压缩到 64 个
x = AveragePooling1D(pool_size=16, padding='valid')(inputs) # 1034 // 16 = 64

# 线性投影：将原始 1 维特征映射到 d_model 维 (32维)
x = Dense(32, kernel_regularizer=l2(1e-4))(x)

# 位置编码（长度正好是 64）
positions = tf.range(start=0, limit=64, delta=1)
pos_emb = Embedding(input_dim=64, output_dim=32)(positions)
x = x + pos_emb

# 轻量级自注意力层 (64x64，CPU跑起来毫无压力)
attn_output = MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
attn_output = Dropout(0.3)(attn_output)
out1 = LayerNormalization(epsilon=1e-6)(x + attn_output) # 残差连接

# 前馈网络层 (FFN)
ffn_output = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(out1)
ffn_output = Dense(32, kernel_regularizer=l2(1e-4))(ffn_output)
ffn_output = Dropout(0.3)(ffn_output)
out2 = LayerNormalization(epsilon=1e-6)(out1 + ffn_output) # 残差连接

# 全局池化与分类头（与 CNN 的 Dense 层设置严格对齐）
x = GlobalAveragePooling1D()(out2)
x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)

outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs=inputs, outputs=outputs)

# 编译参数严格保持 0.0005
model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])

# 5. 训练配置（严格对齐）
early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)

print("\n🚀 开始训练（消融实验 - 轻量级 Transformer 版，CPU跑1轮约几秒）...")
history = model.fit(
    x_train, y_train,
    epochs=150, # 因为有早停，设80足够了
    batch_size=32,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr],
    shuffle=True, verbose=1
)

# 6. 二分类学术指标深度评估 (严格对齐)
print("\n=== 模型深度评估 ===")
y_pred_proba = model.predict(x_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr_val = fp / (fp + tn + 1e-10)
fnr_val = fn / (fn + tp + 1e-10)
g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
mcc = matthews_corrcoef(y_test, y_pred)
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr_curve, tpr_curve)

print("\n" + "="*50)
print(f"准确率 (Accuracy) : {accuracy:.4f}")
print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall)   : {recall:.4f}")
print(f"F1分数 (F1-Score) : {f1:.4f}")
print(f"AUC面积 (ROC-AUC) : {roc_auc:.4f}")
print("-" * 50)
print(f"误报率 (FPR)      : {fpr_val:.4f}")
print(f"漏报率 (FNR)      : {fnr_val:.4f}")
print(f"综合均值 (G-Mean) : {g_mean:.4f}")
print(f"马修斯相关系数    : {mcc:.4f}")
print("="*50)

# 7. 保存结果供画对比图
save_path = r"E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\正式sci论文\二区期刊的sci论文\无泄漏重跑实验\Transformer"
os.makedirs(save_path, exist_ok=True)
pd.DataFrame({'label': y_test, 'pred': y_pred_proba}).to_csv(os.path.join(save_path, "Binary_Transformer.csv"), index=False)
print(f"✅ 轻量级 Transformer 预测结果已成功导出至: {save_path}")

=== 消融实验：轻量级 Transformer 模型 (CPU极速版) ===

1. 加载数据...

🚀 开始训练（消融实验 - 轻量级 Transformer 版，CPU跑1轮约几秒）...
Epoch 1/150
362/362 [==============================] - 9s 18ms/step - loss: 0.4184 - accuracy: 0.8401 - val_loss: 0.3869 - val_accuracy: 0.8514 - lr: 5.0000e-04
Epoch 2/150
362/362 [==============================] - 6s 17ms/step - loss: 0.3968 - accuracy: 0.8596 - val_loss: 0.3751 - val_accuracy: 0.8579 - lr: 5.0000e-04
Epoch 3/150
362/362 [==============================] - 6s 17ms/step - loss: 0.3890 - accuracy: 0.8618 - val_loss: 0.4274 - val_accuracy: 0.8569 - lr: 5.0000e-04
Epoch 4/150
362/362 [==============================] - 6s 17ms/step - loss: 0.3754 - accuracy: 0.8652 - val_loss: 0.4510 - val_accuracy: 0.8434 - lr: 5.0000e-04
Epoch 5/150
362/362 [==============================] - 6s 17ms/step - loss: 0.3736 - accuracy: 0.8674 - val_loss: 0.3776 - val_accuracy: 0.8780 - lr: 5.0000e-04
Epoch 6/150
362/362 [==============================] - 7s 18ms/step - loss: 0.3676 - accuracy: 

In [6]:
# -*- coding: utf-8 -*-
"""
【消融实验专用】轻量级 TCN 模型（CPU极速降采样版，严格参数对齐）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

import matplotlib
matplotlib.rcParams["font.family"] = ["Times New Roman", "SimHei", "Arial"]
matplotlib.rcParams["axes.unicode_minus"] = False
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Conv1D, Add, Activation, AveragePooling1D, GlobalAveragePooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

print("=== 消融实验：轻量级 TCN 模型 (CPU 极速版) ===")

# 2. 数据加载（严格一致）
file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
try:
    df = pd.read_excel(file_path)
except Exception:
    df = pd.read_csv(file_path, encoding='utf-8')

if 'Unnamed: 0' in df.columns:
    df.drop(['Unnamed: 0'], axis=1, inplace=True)
    
Y = df['output'].astype(int).values
X = df.drop(['output'], axis=1).values

scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X).astype(np.float32)

=== 消融实验：轻量级 TCN 模型 ===

1. 加载数据...

🚀 开始训练（消融实验 - 轻量级 TCN 版，CPU 跑1轮仅需几秒）...
Epoch 1/80
298/362 [=======================>......] - ETA: 17s - loss: 0.4323 - accuracy: 0.8405

KeyboardInterrupt: 

In [8]:
# 3. 严格对齐数据集划分 (test_size=0.2, stratify=Y, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42, shuffle=True, stratify=Y
)

x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]
input_dim = x_train.shape[1]

# 4. 构建轻量级 TCN 模型
inputs = Input(shape=(input_dim, 1))

# ⚡ 核心加速：将1034个时间步池化压缩到64个，大幅削减计算量
x = AveragePooling1D(pool_size=16, padding='valid')(inputs) # 1034 // 16 = 64

def tcn_block(input_tensor, filters, kernel_size, dilation_rate):
    conv1 = Conv1D(filters=filters, kernel_size=kernel_size, dilation_rate=dilation_rate, 
                   padding='causal', activation='relu', kernel_regularizer=l2(1e-4))(input_tensor)
    conv1 = BatchNormalization()(conv1)
    conv1 = Dropout(0.3)(conv1)
    
    conv2 = Conv1D(filters=filters, kernel_size=kernel_size, dilation_rate=dilation_rate, 
                   padding='causal', activation='relu', kernel_regularizer=l2(1e-4))(conv1)
    conv2 = BatchNormalization()(conv2)
    conv2 = Dropout(0.3)(conv2)
    
    if input_tensor.shape[-1] != filters:
        shortcut = Conv1D(filters=filters, kernel_size=1, padding='same')(input_tensor)
        shortcut = BatchNormalization()(shortcut)
    else:
        shortcut = input_tensor
        
    output = Add()([shortcut, conv2])
    output = Activation('relu')(output)
    return output

# 缩减通道数至16，层数减为2层，CPU跑得飞快
x = tcn_block(x, filters=16, kernel_size=3, dilation_rate=1)
x = tcn_block(x, filters=16, kernel_size=3, dilation_rate=2)

# 分类头（严格对齐 CNN）
x = GlobalAveragePooling1D()(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(x)
x = Dropout(0.3)(x)

outputs = Dense(1, activation='sigmoid')(x)
model = Model(inputs=inputs, outputs=outputs)

model.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0005), metrics=['accuracy'])

# 5. 训练配置（对齐，batch_size调大加速）
early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)

print("\n🚀 开始训练（CPU极速版，每轮预计仅需1-2秒）...")
history = model.fit(
    x_train, y_train,
    epochs=150, 
    batch_size=32, # 增大batch加速CPU
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr],
    shuffle=True, verbose=1
)

# 6. 评估
print("\n=== 模型深度评估 ===")
y_pred_proba = model.predict(x_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fpr_val = fp / (fp + tn + 1e-10)
fnr_val = fn / (fn + tp + 1e-10)
g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))
mcc = matthews_corrcoef(y_test, y_pred)
fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr_curve, tpr_curve)

print("\n" + "="*50)
print(f"准确率 (Accuracy) : {accuracy:.4f}")
print(f"精确率 (Precision): {precision:.4f}")
print(f"召回率 (Recall)   : {recall:.4f}")
print(f"F1分数 (F1-Score) : {f1:.4f}")
print(f"AUC面积 (ROC-AUC) : {roc_auc:.4f}")
print("-" * 50)
print(f"误报率 (FPR)      : {fpr_val:.4f}")
print(f"漏报率 (FNR)      : {fnr_val:.4f}")
print(f"综合均值 (G-Mean) : {g_mean:.4f}")
print(f"马修斯相关系数    : {mcc:.4f}")
print("="*50)

save_path = r"E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\正式sci论文\二区期刊的sci论文\无泄漏重跑实验\TCN"
os.makedirs(save_path, exist_ok=True)
pd.DataFrame({'label': y_test, 'pred': y_pred_proba}).to_csv(os.path.join(save_path, "Binary_TCN.csv"), index=False)
print(f"✅ 轻量级 TCN 预测结果已成功导出至: {save_path}")


🚀 开始训练（CPU极速版，每轮预计仅需1-2秒）...
Epoch 1/150
362/362 [==============================] - 7s 11ms/step - loss: 0.4595 - accuracy: 0.8174 - val_loss: 0.7416 - val_accuracy: 0.7501 - lr: 5.0000e-04
Epoch 2/150
362/362 [==============================] - 3s 9ms/step - loss: 0.4049 - accuracy: 0.8517 - val_loss: 0.3700 - val_accuracy: 0.8576 - lr: 5.0000e-04
Epoch 3/150
362/362 [==============================] - 4s 10ms/step - loss: 0.3828 - accuracy: 0.8606 - val_loss: 0.3553 - val_accuracy: 0.8662 - lr: 5.0000e-04
Epoch 4/150
362/362 [==============================] - 3s 10ms/step - loss: 0.3726 - accuracy: 0.8683 - val_loss: 0.3530 - val_accuracy: 0.8680 - lr: 5.0000e-04
Epoch 5/150
362/362 [==============================] - 4s 10ms/step - loss: 0.3622 - accuracy: 0.8725 - val_loss: 0.3466 - val_accuracy: 0.8697 - lr: 5.0000e-04
Epoch 6/150
362/362 [==============================] - 3s 10ms/step - loss: 0.3598 - accuracy: 0.8737 - val_loss: 0.3515 - val_accuracy: 0.8638 - lr: 5.0000e-04
Epoch

In [12]:
# -*- coding: utf-8 -*-
"""
第一部分：固定阈值 EMD 数据预处理（全局统一阈值，不强制剔除IMF1）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(42)
np.random.seed(42)

import pandas as pd
from PyEMD import EMD
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class FixedThresholdEMDProcessor:
    def __init__(self, n_imfs=8):
        self.n_imfs = n_imfs
        self.emd = EMD()

    def _calc_correlation(self, orig, recon):
        min_len = min(len(orig), len(recon))
        if min_len < 2: return 0.0
        orig, recon = orig[:min_len], recon[:min_len]
        if np.std(orig) < 1e-6 or np.std(recon) < 1e-6: return 0.0
        return np.corrcoef(orig, recon)[0, 1]

    def _fixed_denoise(self, sample):
        orig_signal = sample.copy()
        try:
            imfs = self.emd(orig_signal, max_imf=self.n_imfs)
            residue = orig_signal - np.sum(imfs, axis=0)
            if len(imfs) < 2: return orig_signal
            
            total_energy = np.sum(orig_signal ** 2) + 1e-10
            energy_ratios = []
            correlations = []
            for imf in imfs:
                energy_ratios.append(np.sum(imf ** 2) / total_energy)
                if np.std(imf) < 1e-6 or np.std(orig_signal) < 1e-6:
                    correlations.append(0)
                else:
                    correlations.append(np.corrcoef(orig_signal, imf)[0, 1])
            energy_ratios = np.array(energy_ratios)
            correlations = np.array(correlations)
            
            # 👇 核心：全局统一固定阈值，不分标签，不强制剔除IMF1
            useful_mask = (energy_ratios > 0.02*0.8) | (np.abs(correlations) > 0.12*0.8)
            useful_imfs = imfs[useful_mask]
            
            # 保底：如果一个都没筛出来，就全保留
            if len(useful_imfs) == 0:
                useful_imfs = imfs
                
            reconstructed = np.sum(useful_imfs, axis=0) + residue
            
            # 统一保底机制（与动态EMD一致）
            if self._calc_correlation(orig_signal, reconstructed) < 0.85:
                useful_mask = (energy_ratios > 0.01) | (np.abs(correlations) > 0.06)
                useful_imfs = imfs[useful_mask]
                reconstructed = np.sum(useful_imfs, axis=0) + residue
                if self._calc_correlation(orig_signal, reconstructed) < 0.85:
                    reconstructed = np.sum(imfs, axis=0) + residue
            
            # 幅值校准
            if np.max(reconstructed) != 0:
                reconstructed = reconstructed * (np.max(orig_signal) / np.max(reconstructed))
            return reconstructed[:len(orig_signal)]
        except Exception:
            return orig_signal

    def process_data(self, X_data):
        print(f"=== 固定阈值 EMD 处理 {len(X_data)} 个样本 ===")
        processed = []
        for idx, sample in enumerate(X_data):
            if idx % 200 == 0: print(f"进度: {idx}/{len(X_data)}")
            processed.append(self._fixed_denoise(sample))
        return np.array(processed)

def main():
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    try:
        df = pd.read_excel(file_path)
    except Exception:
        df = pd.read_csv(file_path, encoding='utf-8')
    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values

    # ✅ 与动态EMD完全对齐：种子42，test_size=0.2
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )

    processor = FixedThresholdEMDProcessor(n_imfs=8)
    X_train = processor.process_data(X_train_raw)
    X_test = processor.process_data(X_test_raw)

    output_dir = "fixed_threshold_emd_data"
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "X_train.npy"), X_train)
    np.save(os.path.join(output_dir, "y_train.npy"), y_train_raw)
    np.save(os.path.join(output_dir, "X_test.npy"), X_test)
    np.save(os.path.join(output_dir, "y_test.npy"), y_test_raw)
    print(f"\n✅ 固定阈值 EMD 预处理完成！数据已保存至: {output_dir}")

if __name__ == "__main__":
    main()

=== 固定阈值 EMD 处理 11568 个样本 ===
进度: 0/11568
进度: 200/11568
进度: 400/11568
进度: 600/11568
进度: 800/11568
进度: 1000/11568
进度: 1200/11568
进度: 1400/11568
进度: 1600/11568
进度: 1800/11568
进度: 2000/11568
进度: 2200/11568
进度: 2400/11568
进度: 2600/11568
进度: 2800/11568
进度: 3000/11568
进度: 3200/11568
进度: 3400/11568
进度: 3600/11568
进度: 3800/11568
进度: 4000/11568
进度: 4200/11568
进度: 4400/11568
进度: 4600/11568
进度: 4800/11568
进度: 5000/11568
进度: 5200/11568
进度: 5400/11568
进度: 5600/11568
进度: 5800/11568
进度: 6000/11568
进度: 6200/11568
进度: 6400/11568
进度: 6600/11568
进度: 6800/11568
进度: 7000/11568
进度: 7200/11568
进度: 7400/11568
进度: 7600/11568
进度: 7800/11568
进度: 8000/11568
进度: 8200/11568
进度: 8400/11568
进度: 8600/11568
进度: 8800/11568
进度: 9000/11568
进度: 9200/11568
进度: 9400/11568
进度: 9600/11568
进度: 9800/11568
进度: 10000/11568
进度: 10200/11568
进度: 10400/11568
进度: 10600/11568
进度: 10800/11568
进度: 11000/11568
进度: 11200/11568
进度: 11400/11568
=== 固定阈值 EMD 处理 2893 个样本 ===
进度: 0/2893
进度: 200/2893
进度: 400/2893
进度: 600/2893
进度: 800/2893
进度: 100

In [13]:
# -*- coding: utf-8 -*-
"""
第二部分：固定阈值 EMD-SE-CNN 二分类网络训练与预测
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import random
import numpy as np
import tensorflow as tf
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=3, **kwargs):
        super(SEBlock, self).__init__(**kwargs)
        self.reduction = reduction
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False, kernel_initializer='he_normal')
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False, kernel_initializer='he_normal')
        super(SEBlock, self).build(input_shape)
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs)
        se = Reshape((1, self.channels))(se)
        se = self.fc1(se)
        se = self.fc2(se)
        return Multiply()([inputs, se])
    def get_config(self):
        config = super(SEBlock, self).get_config()
        config.update({'reduction': self.reduction})
        return config

def main():
    print("=== 第二阶段：固定阈值 EMD-SE-CNN 二分类窃电检测 ===")
    data_dir = "fixed_threshold_emd_data"
    try:
        X_train = np.load(os.path.join(data_dir, "X_train.npy"))
        y_train = np.load(os.path.join(data_dir, "y_train.npy"))
        X_test = np.load(os.path.join(data_dir, "X_test.npy"))
        y_test = np.load(os.path.join(data_dir, "y_test.npy"))
        X_train = X_train[..., np.newaxis] 
        X_test = X_test[..., np.newaxis]   
        print(f"✅ 数据加载成功！训练集: {X_train.shape}, 测试集: {X_test.shape}")
    except FileNotFoundError:
        print("❌ 找不到数据文件夹，请先运行第一部分预处理脚本！")
        return

    inputs = Input(shape=X_train.shape[1:])
    x = Conv1D(32, 5, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(2)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Conv1D(64, 3, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(4)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x); x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x); x = Dropout(0.3)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0005), metrics=['accuracy'])

    early_stop = EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, verbose=1)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-7, verbose=1)

    print("\n🚀 开始模型训练...")
    history = model.fit(
        X_train, y_train,
        epochs=150, batch_size=32,
        validation_data=(X_test, y_test),
        callbacks=[early_stop, reduce_lr],
        shuffle=True, verbose=1
    )

    print("\n=== 模型硬核指标评估 ===")
    y_pred_proba = model.predict(X_test, verbose=0).flatten()
    y_pred = (y_pred_proba > 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    fpr_curve, tpr_curve, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr_curve, tpr_curve)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    fpr_val = fp / (fp + tn + 1e-10)
    fnr_val = fn / (fn + tp + 1e-10)
    g_mean = np.sqrt((tp / (tp + fn + 1e-10)) * (tn / (tn + fp + 1e-10)))

    print("\n" + "="*50)
    print("=== SCI 级二分类详细性能指标 (固定阈值EMD) ===")
    print("="*50)
    print(f"准确率 (Accuracy)       : {accuracy:.4f}")
    print(f"精确率 (Precision)      : {precision:.4f}")
    print(f"召回率 (Recall)         : {recall:.4f}")
    print(f"F1分数 (F1-Score)       : {f1:.4f}")
    print(f"AUC面积 (ROC-AUC)       : {roc_auc:.4f}")
    print("-" * 50)
    print(f"误报率 (FPR)            : {fpr_val:.4f}")
    print(f"漏报率 (FNR)            : {fnr_val:.4f}")
    print(f"综合均值 (G-Mean)       : {g_mean:.4f}")
    print(f"马修斯系数 (MCC)        : {mcc:.4f}")
    print("="*50)

    save_path = "Binary_Results_CSV_Fixed_EMD"
    os.makedirs(save_path, exist_ok=True)
    df_save = pd.DataFrame({'label': y_test, 'pred': y_pred_proba})
    csv_filename = os.path.join(save_path, "Binary_Fixed_EMD_CNN_SE.csv")
    df_save.to_csv(csv_filename, index=False)
    print(f"\n✅ 预测结果已成功导出至: {csv_filename}")

if __name__ == "__main__":
    main()

=== 第二阶段：固定阈值 EMD-SE-CNN 二分类窃电检测 ===
✅ 数据加载成功！训练集: (11568, 1034, 1), 测试集: (2893, 1034, 1)

🚀 开始模型训练...
Epoch 1/150
362/362 [==============================] - 38s 100ms/step - loss: 0.4648 - accuracy: 0.8486 - val_loss: 0.6736 - val_accuracy: 0.8012 - lr: 5.0000e-04
Epoch 2/150
362/362 [==============================] - 35s 98ms/step - loss: 0.4174 - accuracy: 0.8774 - val_loss: 0.3967 - val_accuracy: 0.8845 - lr: 5.0000e-04
Epoch 3/150
362/362 [==============================] - 40s 110ms/step - loss: 0.3986 - accuracy: 0.8873 - val_loss: 0.3781 - val_accuracy: 0.8928 - lr: 5.0000e-04
Epoch 4/150
362/362 [==============================] - 37s 102ms/step - loss: 0.3721 - accuracy: 0.8958 - val_loss: 0.4024 - val_accuracy: 0.9036 - lr: 5.0000e-04
Epoch 5/150
362/362 [==============================] - 36s 99ms/step - loss: 0.3693 - accuracy: 0.8983 - val_loss: 0.3890 - val_accuracy: 0.8991 - lr: 5.0000e-04
Epoch 6/150
362/362 [==============================] - 36s 99ms/step - loss: 0.3563 

In [15]:
# -*- coding: utf-8 -*-
"""
参数敏感性分析（修正版：加入了归一化，绝对准确）
测试 min_corr（0.80, 0.85, 0.90）和 relax（0.6, 0.8, 1.0）
"""
import os
os.environ['PYTHONHASHSEED'] = '42'
import random
import numpy as np
random.seed(42)
np.random.seed(42)

import pandas as pd
from PyEMD import EMD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, matthews_corrcoef
from sklearn.metrics import precision_score, recall_score
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D, Dropout, BatchNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D, Reshape, Multiply, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow as tf
tf.random.set_seed(42)
import warnings
warnings.filterwarnings('ignore')

class SEBlock(tf.keras.layers.Layer):
    def __init__(self, reduction=3, **kwargs):
        super().__init__(**kwargs); self.reduction = reduction
    def build(self, input_shape):
        self.channels = input_shape[-1]
        self.fc1 = Dense(max(self.channels // self.reduction, 4), activation='relu', use_bias=False)
        self.fc2 = Dense(self.channels, activation='sigmoid', use_bias=False)
        super().build(input_shape)
    def call(self, inputs):
        se = GlobalAveragePooling1D()(inputs); se = Reshape((1, self.channels))(se)
        se = self.fc1(se); se = self.fc2(se)
        return Multiply()([inputs, se])

def build_model(dim):
    inputs = Input(shape=(dim, 1))
    x = Conv1D(32, 5, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(inputs)
    x = SEBlock(3)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Conv1D(64, 3, padding='same', activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = SEBlock(3)(x); x = BatchNormalization()(x); x = MaxPooling1D(2, padding='same')(x); x = Dropout(0.3)(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(2e-4))(x)
    x = BatchNormalization()(x); x = Dropout(0.5)(x)
    x = Dense(64, activation='relu', kernel_regularizer=l2(2e-4))(x); x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(loss='binary_crossentropy', optimizer=Adam(0.0005), metrics=['accuracy'])
    return model

def adaptive_emd(sample, is_theft, min_corr, relax):
    orig = sample.copy()
    try:
        imfs = EMD()(orig, max_imf=8)
        if len(imfs) < 2: return orig
        res = orig - np.sum(imfs, axis=0)
        total_e = np.sum(orig**2) + 1e-10
        e_r = np.array([np.sum(i**2)/total_e for i in imfs])
        corr = np.array([np.corrcoef(orig, i)[0,1] if np.std(i)>1e-6 else 0 for i in imfs])
        
        if is_theft:
            mask = (e_r > 0.02 * relax) | (np.abs(corr) > 0.12 * relax)
        else:
            mask = (e_r > 0.02) | (np.abs(corr) > 0.12)
        useful = imfs[mask]
        if not is_theft and len(useful) > 1: useful = useful[1:]
        recon = np.sum(useful, axis=0) + res
        
        if np.corrcoef(orig[:len(recon)], recon)[0,1] < min_corr:
            mask2 = (e_r > 0.01) | (np.abs(corr) > 0.06)
            useful = imfs[mask2]; recon = np.sum(useful, axis=0) + res
            if np.corrcoef(orig[:len(recon)], recon)[0,1] < min_corr:
                recon = np.sum(imfs, axis=0) + res
        if np.max(recon) != 0: recon = recon * (np.max(orig) / np.max(recon))
        return recon[:len(orig)]
    except: return orig

def main():
    file_path = r'E:\阳光2\反窃电论文\最新高水平窃电sci\聚合相关\xin\Electricity-Theft-Detection--master\sample.xlsx'
    try:
        df = pd.read_excel(file_path)
    except Exception:
        df = pd.read_csv(file_path, encoding='utf-8')
    if 'Unnamed: 0' in df.columns:
        df.drop(['Unnamed: 0'], axis=1, inplace=True)
    Y = df['output'].values
    X = df.drop(['output'], axis=1).values

    # 👇 核心修复1：先切分数据，再做归一化（防止数据泄露）
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, Y, test_size=0.2, random_state=42, stratify=Y
    )
    
    # 归一化！这是之前漏掉的最关键一步！
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_test_scaled = scaler.transform(X_test_raw).astype(np.float32)

    # 👇 核心修复2：取子集加速，但保证训练集有足够样本
    X_tr_sub, _, y_tr_sub, _ = train_test_split(
        X_train_scaled, y_train, train_size=3000, random_state=42, stratify=y_train
    )
    X_te_sub, _, y_te_sub, _ = train_test_split(
        X_test_scaled, y_test, train_size=1000, random_state=42, stratify=y_test
    )

    configs = [
        ('min_corr=0.80', 0.80, 0.8),
        ('min_corr=0.85(默认)', 0.85, 0.8),
        ('min_corr=0.90', 0.90, 0.8),
        ('relax=0.6', 0.85, 0.6),
        ('relax=0.8(默认)', 0.85, 0.8),
        ('relax=1.0', 0.85, 1.0),
    ]

    results = []
    print("="*60)
    print("参数敏感性分析开始（修复归一化问题）")
    print("="*60)

    for name, mc, rl in configs:
        print(f"\n[Running] {name} ...")
        X_tr = np.array([adaptive_emd(s, l==1, mc, rl) for s, l in zip(X_tr_sub, y_tr_sub)])[..., np.newaxis]
        X_te = np.array([adaptive_emd(s, l==1, mc, rl) for s, l in zip(X_te_sub, y_te_sub)])[..., np.newaxis]

        model = build_model(X_tr.shape[1])
        early_stop = EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True, verbose=0)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=0)
        model.fit(X_tr, y_tr_sub, epochs=50, batch_size=32, validation_data=(X_te, y_te_sub),
                  callbacks=[early_stop, reduce_lr], shuffle=True, verbose=0)

        proba = model.predict(X_te, verbose=0).flatten()
        pred = (proba > 0.5).astype(int)

        acc = accuracy_score(y_te_sub, pred)
        f1 = f1_score(y_te_sub, pred, zero_division=0)
        auc = roc_auc_score(y_te_sub, proba)
        mcc = matthews_corrcoef(y_te_sub, pred)
        prec = precision_score(y_te_sub, pred, zero_division=0)
        rec = recall_score(y_te_sub, pred, zero_division=0)

        results.append({
            'Setting': name, 'min_corr': mc, 'relax': rl,
            'Accuracy': round(acc,4), 'Precision': round(prec,4),
            'Recall': round(rec,4), 'F1': round(f1,4),
            'AUC': round(auc,4), 'MCC': round(mcc,4)
        })
        print(f"   -> Acc: {acc:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}, MCC: {mcc:.4f}")

    df_res = pd.DataFrame(results)
    df_res.to_csv('sensitivity_analysis_results.csv', index=False, encoding='utf-8-sig')
    print("\n" + "="*60)
    print("✅ 参数敏感性分析完成！结果已保存至 sensitivity_analysis_results.csv")
    print("="*60)
    print(df_res.to_string(index=False))

if __name__ == "__main__":
    main()

参数敏感性分析开始（修复归一化问题）

[Running] min_corr=0.80 ...
   -> Acc: 0.9470, F1: 0.8946, AUC: 0.9851, MCC: 0.8593

[Running] min_corr=0.85(默认) ...
   -> Acc: 0.9320, F1: 0.8601, AUC: 0.9764, MCC: 0.8158

[Running] min_corr=0.90 ...
   -> Acc: 0.9100, F1: 0.8193, AUC: 0.9488, MCC: 0.7594

[Running] relax=0.6 ...
   -> Acc: 0.9360, F1: 0.8678, AUC: 0.9747, MCC: 0.8264

[Running] relax=0.8(默认) ...
   -> Acc: 0.9350, F1: 0.8713, AUC: 0.9749, MCC: 0.8279

[Running] relax=1.0 ...
   -> Acc: 0.9340, F1: 0.8675, AUC: 0.9759, MCC: 0.8235

✅ 参数敏感性分析完成！结果已保存至 sensitivity_analysis_results.csv
          Setting  min_corr  relax  Accuracy  Precision  Recall     F1    AUC    MCC
    min_corr=0.80      0.80    0.8     0.947     0.8893   0.900 0.8946 0.9851 0.8593
min_corr=0.85(默认)      0.85    0.8     0.932     0.8856   0.836 0.8601 0.9764 0.8158
    min_corr=0.90      0.90    0.8     0.910     0.8226   0.816 0.8193 0.9488 0.7594
        relax=0.6      0.85    0.6     0.936     0.8974   0.840 0.8678 0.9747 0.82